# Notebook 03: Self-Attention

**Module:** 10 Transformers  
**Duration:** ~2.5 hrs

---

## Learning Objectives

1. Derive self-attention with Q=K=V from input
2. Track matrix dimensions through layers
3. Implement self-attention in PyTorch
4. Compute O(n²) complexity

## Self-Attention

**Self-attention:** Q, K, V all derived from the **same** input sequence X.

$$Q = XW_Q, \quad K = XW_K, \quad V = XW_V$$

**Input:** $X \in \mathbb{R}^{\text{seq} \times d_{model}}$

**Output:** Same shape as X — each token replaced by weighted mix of all tokens.

**Complexity:** $O(n^2 \cdot d)$ for sequence length $n$ — bottleneck for long sequences.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

plt.rcParams['figure.figsize'] = (8, 5)
torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
def scaled_dot_product_attention_torch(Q, K, V, mask=None):
    d_k = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    weights = F.softmax(scores, dim=-1)
    return torch.matmul(weights, V), weights

In [ ]:
class SelfAttention(nn.Module):
    def __init__(self, d_model, d_k):
        super().__init__()
        self.W_q = nn.Linear(d_model, d_k, bias=False)
        self.W_k = nn.Linear(d_model, d_k, bias=False)
        self.W_v = nn.Linear(d_model, d_k, bias=False)
    def forward(self, x):
        Q, K, V = self.W_q(x), self.W_k(x), self.W_v(x)
        out, w = scaled_dot_product_attention_torch(Q, K, V)
        return out, w

x = torch.randn(2, 6, 64)  # batch, seq, d_model
sa = SelfAttention(64, 32)
out, w = sa(x)
print(f'Input: {x.shape} -> Output: {out.shape}, Weights: {w.shape}')

## Dimension Cheat Sheet

| Tensor | Shape |
|--------|-------|
| X | (batch, seq, d_model) |
| Q, K | (batch, seq, d_k) |
| V | (batch, seq, d_v) |
| Weights | (batch, seq, seq) |
| Output | (batch, seq, d_v) |

---

## Summary

Self-attention: each token attends to all tokens; Q,K,V from same input.

**Next:** [04_Multi_Head_Attention.ipynb](04_Multi_Head_Attention.ipynb)